# EHS Accident Classification — Pipeline Analysis & Research Plan

This notebook documents:
1. Analysis of the existing zero-shot pipeline (`zero shot pipeline.ipynb`)
2. Required API keys and setup
3. Technical summary of the current workflow
4. Proposed modular project structure
5. Research directions and improvements
6. Experiment logging strategy

> **Status**: Analysis-only notebook. No code is re-implemented here — it documents the plan before implementation begins.

---
## Step 1 — Current Pipeline Analysis

### Platform
- Originally designed for **Google Colab** (Google Drive mounting, Colab file upload cells)
- Now being migrated to a local VS Code workspace

### Model
- **Vertex AI Gemini** family, specifically:
  - `gemini-1.5-pro` (early test)
  - `gemini-2.0-flash` (intermediate)
  - `gemini-2.5-flash` (current — all production attempts)
- Project ID: `cultivated-cove-488416-e5`
- Location: `us-central1`

### Video Storage
- 73 animated workplace safety videos stored in **Google Cloud Storage**
- Bucket: `ehs-video-analysis-2026-pavithran`
- Prefix: `videos/`
- Videos are the `original.mp4` files from an augmented dataset on Google Drive

### Evolution of Attempts

| Attempt | Key Change | Notes |
|---------|------------|-------|
| 1 | Basic two-stage (binary → classify) | `temperature=0.1/0.2`, No threshold |
| 2 | Added confidence threshold `0.6` + stricter binary prompt | Reduces false positives |
| 3 (iter 1) | Frame fallback: 5 frames at 1fps when Stage 1 says No Accident | Improves recall |
| 3 (iter 2) | Frame fallback upgraded to 10 frames at 2fps | More thorough scan |

### Libraries Used

| Library | Purpose |
|---------|----------|
| `google-cloud-aiplatform` / `vertexai` | Vertex AI SDK (Gemini API) |
| `google-cloud-storage` | GCS bucket access |
| `pandas`, `numpy` | Data processing, result storage |
| `scikit-learn` | Accuracy, F1, confusion matrix |
| `seaborn`, `matplotlib` | Confusion matrix plots |
| `moviepy` | Video trimming |
| `yt-dlp` | YouTube video download |
| `ffmpeg` (system) | Frame extraction via subprocess |
| `openpyxl` | Excel output |

### Where Latency and Cost Occur

| Stage | API Calls Per Video | Notes |
|-------|--------------------|---------|
| Stage 1 binary | 3 calls (majority vote) | `gemini-2.5-flash`, temp=0.0 |
| Stage 2 classification | 1 call | Only if Stage 1 → Accident |
| Frame fallback (if needed) | Up to 10 image calls | Downloads video locally first |
| `time.sleep(1)` | +1s per video | Rate limiting |

**Current throughput**: ~12 seconds for a 6-second clip → far below 1:1 real-time target.

**Primary latency causes**:
- 3× redundant Stage 1 calls with `any(votes)` logic (worst case: all 3 calls needed)
- Frame fallback downloads full video to disk then calls Gemini 10× with images
- `gemini-2.5-flash` cold start + video encoding time on Vertex

### External API Dependencies
1. **Google Vertex AI** — Gemini model inference
2. **Google Cloud Storage** — Video asset storage
3. **Google Drive** (via Colab) — Source dataset

---
## Step 2 — Required API Keys and Setup Guide

### Services Required

| Service | What It Does | Auth Method |
|---------|-------------|-------------|
| Google Vertex AI | Gemini model inference | Service Account JSON |
| Google Cloud Storage | Video upload/download | Same service account |

---

### Step A — Create a Google Cloud Service Account

1. Go to [console.cloud.google.com](https://console.cloud.google.com)
2. Select project `cultivated-cove-488416-e5` (or your project)
3. Navigate to **IAM & Admin → Service Accounts**
4. Create a new service account with roles:
   - `Vertex AI User`
   - `Storage Object Admin` (or `Storage Admin`)
5. Download the JSON key file

---

### Step B — Store Credentials Safely

**DO NOT commit the JSON key file to git.**

Create a `.env` file in the project root:

```bash
# .env
GOOGLE_APPLICATION_CREDENTIALS=/absolute/path/to/your-service-account-key.json
GCP_PROJECT_ID=cultivated-cove-488416-e5
GCP_LOCATION=us-central1
GCS_BUCKET_NAME=ehs-video-analysis-2026-pavithran
GCS_VIDEO_PREFIX=videos/
VERTEX_MODEL=gemini-2.5-flash
CONFIDENCE_THRESHOLD=0.6
```

Add `.env` and `*.json` to `.gitignore`.

---

### Step C — Load Keys in Code

```python
import os
from dotenv import load_dotenv

load_dotenv()  # loads .env file

PROJECT_ID  = os.environ["GCP_PROJECT_ID"]
LOCATION    = os.environ["GCP_LOCATION"]
BUCKET_NAME = os.environ["GCS_BUCKET_NAME"]
MODEL_NAME  = os.environ.get("VERTEX_MODEL", "gemini-2.5-flash")

# GOOGLE_APPLICATION_CREDENTIALS is picked up automatically
# by the google-auth library — no manual loading needed.
```

---

### Step D — Python Dependencies

```bash
pip install -r requirements.txt
```

Full `requirements.txt` is in the project root (see Step 4).

System dependency (Linux):

```bash
sudo apt-get install -y ffmpeg
```

---
## Step 3 — Technical Summary of Current Pipeline

### Video Acquisition
- 73 animated workplace safety clips stored in GCS bucket
- Originally sourced from YouTube (via `yt-dlp`) and trimmed with `moviepy`
- Videos are `original.mp4` files from a Google Drive augmented dataset
- Filenames follow pattern: `VID{number}_<description>_original.mp4`

### Stage 1 — Binary Accident Detection
- Model: `gemini-2.5-flash` via Vertex AI
- Input: video URI from GCS (`gs://bucket/...`)
- Prompt instructs model to classify as `incident_detected: true/false`
- Run **3 times** with `temperature=0.0`; result is `any(votes)` (high-recall)
- Confidence gated at threshold `0.6`
- Prompt is **animated-aware**: explicitly tells model the content is animated safety training material

### Frame Fallback (Stage 1.5)
- Triggered when all 3 Stage 1 votes → No Accident
- Video downloaded locally, `ffmpeg` extracts up to 10 frames at 2fps
- Each frame sent to Gemini as image with a simpler accident-detection prompt
- Any single frame returning `incident_detected: true` overrides Stage 1

### Stage 2 — Classification
- Model: `gemini-2.5-flash`, `temperature=0.0`
- Prompt asks for EXACTLY ONE category from 11 classes:
  ```
  Arc Flash, Caught In Machine, Electrocution, Fall, Fire,
  Gas Inhalation, Lifting, Slip, Struck by Object, Trip, Vehicle Incident
  ```
- Returns JSON: `category`, `incident_start_time`, `incident_end_time`, `description`, `root_cause_analysis`
- Invalid categories fall back to `"Fall"`

### Post-processing
- JSON extracted from response with regex stripping of markdown code fences
- `safe_parse_json()` handles list/dict variants
- Exponential backoff retry wrapper (`generate_with_retry`)
- Results collected in a list → `pd.DataFrame` → exported to Excel
- Evaluation: merged with ground-truth `mapping file.xlsx` on `VID{n}` key
- Metrics: binary (accuracy, precision, recall, F1) + multi-class confusion matrix

### Known Issues
- Fallback to `"Fall"` category when parse fails or category not in list — introduces bias
- 3× Stage 1 calls is expensive; `any(votes)` is high-recall but low-precision
- Frame extraction downloads the full video, adding I/O latency
- No latency or cost logging in current implementation
- No experiment versioning — results overwrite previous Excel files

---
## Step 4 — Proposed Modular Project Structure

```
Capstone/
├── .env                          # Local secrets (gitignored)
├── .env.example                  # Template to commit
├── .gitignore
├── README.md
├── requirements.txt
│
├── config/
│   ├── __init__.py
│   ├── categories.py             # MERGED_CATEGORIES, severity order, near-miss list
│   └── settings.py               # Load .env, dataclass Config
│
├── pipeline/
│   ├── __init__.py
│   ├── client.py                 # Vertex AI + GCS client initialization
│   ├── ingestion.py              # GCS listing, download, yt-dlp helpers
│   ├── detection.py              # Stage 1: binary detection (single call or majority vote)
│   ├── frame_fallback.py         # ffmpeg extraction + frame-level Gemini calls
│   ├── classification.py         # Stage 2: multi-class classification
│   ├── structured_output.py      # NEW: decomposed + severity-ordered predictions
│   ├── near_miss.py              # NEW: near-miss detection module
│   └── postprocessing.py         # JSON parsing, normalization, fallback logic
│
├── experiments/
│   ├── __init__.py
│   ├── runner.py                 # ExperimentRunner: param sweeps, batch execution
│   ├── sampling.py               # NEW: top_k/top_p sweep → probabilistic candidates
│   ├── two_stage.py              # NEW: cheap-model Stage 1 + strong-model Stage 2
│   ├── multi_agent.py            # NEW: judge/jury verification pipeline
│   ├── ablation.py               # Compare prompts, models, thresholds
│   └── configs/
│       ├── attempt1.yaml         # Baseline config
│       ├── attempt2.yaml         # Threshold config  
│       ├── attempt3.yaml         # Frame fallback config
│       └── experiment_template.yaml
│
├── evaluation/
│   ├── __init__.py
│   ├── metrics.py                # Binary + multi-class metrics
│   ├── confusion.py              # Confusion analysis + failure diagnostics
│   ├── visualize.py              # Confusion matrix, ROC, latency plots
│   └── ehs_report.py             # NEW: auto-populate EHS incident report fields
│
├── logging/
│   ├── __init__.py
│   └── experiment_logger.py      # Log predictions, latency, cost, params → JSON/CSV
│
├── outputs/
│   └── .gitkeep                  # Results land here (gitignored except .gitkeep)
│
└── notebooks/
    ├── capstone_pipeline_analysis.ipynb   # This notebook
    └── zero shot pipeline.ipynb           # Original (reference only)
```

### Design Principles
- **Config-driven**: every experiment is a YAML config — model, prompt, threshold, sampling params
- **Immutable outputs**: each run writes to a timestamped subfolder under `outputs/`
- **Logged by default**: every API call records latency and token count
- **Decoupled stages**: detection, classification, evaluation are independent; swap any stage
- **Reproducible**: configs + seeds → deterministic results

---
## Step 5 — Research Directions

### Direction 1 — Latency Optimization (Target: 1:1 real-time)

Current bottleneck: 12s for a 6s clip.

| Strategy | Expected Gain | Tradeoff |
|----------|--------------|----------|
| Single Stage 1 call (no 3× voting) | ~2-3× speedup | Lower recall |
| Replace frame fallback with single composite image (montage) | Eliminates download + 10 calls | Less granular |
| Use `gemini-2.0-flash` instead of `2.5-flash` | Faster, cheaper | Slightly lower accuracy |
| Cache GCS URIs (avoid re-upload) | Minor I/O gain | — |
| Async/concurrent processing | Linear to concurrent | Rate limit risk |

---

### Direction 2 — Structured Accident Outputs

Instead of a single label, return decomposed components:

```python
{
  "accident_components": ["Trip", "Fall"],   # ordered: cause → effect
  "severity_order": ["Fall", "Trip"],          # ordered by injury severity
  "primary": "Fall",
  "near_miss": false
}
```

Severity ranking (proposed):
```python
SEVERITY_ORDER = [
    "Arc Flash", "Electrocution", "Fire", "Gas Inhalation",
    "Caught In Machine", "Vehicle Incident", "Struck by Object",
    "Fall", "Lifting", "Slip", "Trip"
]
```

---

### Direction 3 — Two-Stage Model Pipeline

```
Stage 1 (cheap/local):  gemini-2.0-flash OR local vision model
    → High-recall candidate types: {"fall": 0.7, "trip": 0.5, "slip": 0.2}

Stage 2 (strong):  gemini-2.5-pro OR gemini-2.5-flash
    → Verify candidates, select primary, return structured output
```

This separates cost from accuracy: Stage 1 is fast and cheap; Stage 2 runs only when needed.

---

### Direction 4 — Sampling Experiments (top_k / top_p)

Run experiments varying `top_k` and `top_p` to produce **probabilistic candidates**:

```python
{
  "electrocution": 0.50,
  "fall":          0.20,
  "trip":          0.10
}
```

**Heuristic post-processing**:
- Threshold: keep candidates with `p >= 0.15`
- If top-1 probability > 0.5: single-label output
- Otherwise: multi-label output ordered by probability

Experiment grid:
```python
top_k_values = [1, 5, 10, 20, 40]
top_p_values = [0.5, 0.7, 0.9, 0.95, 1.0]
```

---

### Direction 5 — Multi-Agent Validation

```
Agent 1 (Classifier):  gemini-2.5-flash → initial prediction
Agent 2 (Verifier):    gemini-2.5-flash → "Do you agree? If not, what is correct?"
Agent 3 (Judge):       gemini-2.5-pro  → final arbitration (only on disagreement)
```

Cost control: run Agent 3 only when Agents 1 and 2 disagree.

---

### Direction 6 — Near-Miss Classification

Extend the binary output:
```python
{
  "incident_type": "near_miss | accident | safe",
  "near_miss_category": "potential_fall | potential_electrocution | ..."
}
```

Near-miss definition for prompts:
> A situation where the human character was at risk but did NOT make contact with the hazard and did NOT fall/collapse.

---

### Direction 7 — Model Diagnostics

Log for every prediction:
- full prompt text
- raw model response
- parsed output
- ground truth label
- correct/incorrect flag
- error type (false positive, false negative, misclassification)
- latency (seconds)
- token count (input/output)
- estimated cost

Failure analysis:
- Per-class F1 tracking across experiments
- Confusion pair analysis: which categories get confused most often?
- Low-confidence prediction review

---

### Direction 8 — Ablation Studies

Systematic comparison dimensions:

| Axis | Options |
|------|---------|
| Prompt | baseline, animated-aware, strict, lenient |
| Model | gemini-2.0-flash, gemini-2.5-flash, gemini-2.5-pro |
| Temperature | 0.0, 0.1, 0.2, 0.5 |
| top_k | 1, 5, 20, 40 |
| Voting | 1×, 3× any, 3× majority |
| Threshold | 0.5, 0.6, 0.7, 0.8 |
| Frame fallback | none, 5 frames, 10 frames, composite |

Each experiment config is a YAML file. The runner sweeps all combinations and logs results.

---

### Direction 9 — Action Recognition Models

Candidate 3D/action recognition models to evaluate:

| Model | Type | Notes |
|-------|------|-------|
| `TimeSformer` (Hugging Face) | Transformer-based video | `facebook/timesformer-base-finetuned-k400` |
| `VideoMAE` | Masked autoencoder video | `MCG-NJU/videomae-base-finetuned-kinetics` |
| `X3D` (torchvision) | 3D CNN | Lightweight, fast |
| `CLIP` + frame sampling | Image-text matching | Zero-shot via text labels |

Evaluation script will:
1. Run inference on the 73-video dataset
2. Map Kinetics action labels → our 11 accident categories
3. Compute same binary/multi-class metrics
4. Document failure modes with examples

---

### Direction 10 — EHS Report Auto-Population (Optional)

Structured output designed to map to standard EHS incident report fields:

```python
{
  "incident_type": "Fall",
  "incident_components": ["Trip", "Fall"],
  "severity": "High",
  "near_miss": False,
  "timestamp": "00:00:03",
  "location_description": "warehouse floor near shelving",
  "root_cause": "Obstruction in walkway caused loss of footing",
  "corrective_actions": ["Mark walkway hazards", "Conduct housekeeping audit"],
  "ehs_category": "Struck Against",   # OSHA classification
  "osha_recordable": True
}
```

---
## Step 6 — Experiment Logging Schema

Every experiment run exports to `outputs/<timestamp>_<experiment_name>/`:

### `predictions.jsonl` — One record per video
```json
{
  "run_id": "20260312_142301_attempt1",
  "video_id": "VID001_slip_original",
  "gcs_uri": "gs://bucket/videos/VID001.mp4",
  "stage1_votes": [true, true, false],
  "stage1_detected": true,
  "stage1_confidence": 0.82,
  "frame_fallback_used": false,
  "predicted_category": "Slip",
  "predicted_components": ["Slip"],
  "true_category": "Slip",
  "correct": true,
  "description": "Worker loses footing on wet floor",
  "root_cause_analysis": "Wet surface without warning signage",
  "stage1_latency_s": 3.2,
  "stage2_latency_s": 4.1,
  "total_latency_s": 7.3,
  "stage1_input_tokens": 1200,
  "stage1_output_tokens": 40,
  "stage2_input_tokens": 1500,
  "stage2_output_tokens": 120,
  "estimated_cost_usd": 0.0018,
  "model": "gemini-2.5-flash",
  "temperature": 0.0,
  "top_k": null,
  "top_p": null
}
```

### `metrics.json` — Aggregate evaluation
```json
{
  "run_id": "20260312_142301_attempt1",
  "config": "configs/attempt1.yaml",
  "n_videos": 73,
  "n_failed": 2,
  "binary_accuracy": 0.87,
  "binary_precision": 0.91,
  "binary_recall": 0.85,
  "binary_f1": 0.88,
  "multiclass_accuracy": 0.72,
  "per_class_f1": {
    "Fall": 0.80, "Slip": 0.65, "Trip": 0.70,
    "Electrocution": 0.90, "...": "..."
  },
  "mean_latency_s": 8.4,
  "total_cost_usd": 0.14,
  "timestamp": "2026-03-12T14:23:01"
}
```

### `config_snapshot.yaml` — Exact config used
A copy of the YAML config file used for this run, so any result is reproducible.

### `confusion_matrix.csv` — Raw CM values for offline plotting

### `failed_videos.csv` — Error log

---
## Implementation Order

Now that the analysis is complete, implementation will proceed in this order:

1. **`config/`** — `settings.py` (env loading), `categories.py` (labels + severity)
2. **`pipeline/client.py`** — Vertex AI + GCS initialization
3. **`pipeline/postprocessing.py`** — `safe_parse_json`, retry wrapper
4. **`pipeline/detection.py`** — Stage 1 binary detection
5. **`pipeline/classification.py`** — Stage 2 classification
6. **`pipeline/frame_fallback.py`** — ffmpeg + frame classification
7. **`logging/experiment_logger.py`** — latency + cost logging
8. **`evaluation/metrics.py`** — binary + multi-class metrics
9. **`experiments/runner.py`** — experiment orchestration
10. **`pipeline/structured_output.py`** — decomposed + severity-ordered outputs
11. **`experiments/sampling.py`** — top_k/top_p sweeps
12. **`experiments/multi_agent.py`** — judge/jury pipeline
13. **`pipeline/near_miss.py`** — near-miss classification
14. **`evaluation/ehs_report.py`** — EHS report structure

Each module will be implemented and tested individually before integration.